# 19 · OCR cache cho ViVQA — chuẩn bị adapter dùng chung S2 (W11, bước 3)

**Vấn đề:** Adapter dùng chung (W11) chọn phương án **S2**: **mọi câu hỏi** của cả hai dataset đều đi qua đúng một luật prompt, giống prototype thật. Luật đó là chế độ `ocr` + VietOCR · template v1 · cắt 600 ký tự. Ảnh không có chữ thì prompt **tự trùng hệt** zero-shot (`src/prompting/builder.py`). ViTextVQA đã có OCR cache (W08–W10). **ViVQA chưa có**, nên notebook này tạo cache đó.

**Notebook này làm:**
1. Cài PaddleOCR + VietOCR **cùng phiên bản** với cache ViTextVQA (mục 1–3).
2. Chạy thử 100 ảnh test để đo tốc độ và xem tận mắt (mục 4).
3. OCR toàn bộ ảnh `train.json` + `test.json` của ViVQA (mục 5). Có thể chạy tiếp nếu bị ngắt.
4. Kiểm tra: đủ ảnh, lỗi ≤ 1%, engine **trùng** cache ViTextVQA. Sau đó thống kê tỉ lệ câu có dòng OCR và tỉ lệ "đáp án nằm trong OCR" trên test (mục 6).
5. Ghi biên bản `experiments/W11_ocr_cache_vivqa/` để tái lập (mục 7).

**Runtime:** **T4 GPU** (Runtime → Change runtime type → T4 GPU). OCR không cần L4. Cache chỉ tạo một lần, mọi run sau chỉ đọc lại.

**Không** cài `transformers`, **không** nạp Qwen ở notebook này.

## 1. Thiết lập môi trường

Gắn Google Drive, lấy code mới nhất từ GitHub, in loại GPU.

*Kỳ vọng: commit mới nhất (có `src/eval/full_eval_table.py` của W11), `GPU: Tesla T4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/full_eval_table.py'), 'Code cũ -> git pull chưa lấy được commit W11 bước 1-3'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU (OCR sẽ chạy CPU, chậm hơn nhiều)')
print('✅ Thiết lập xong')

## 2. Cài PaddleOCR 3.7 (thử GPU trước, không được thì tự chuyển CPU) — **torch phải còn nguyên**

**Vấn đề (gặp thật ở lần chạy 13/10):** `paddlepaddle-gpu` bản CUDA 12.6 cài kèm thư viện NCCL của CUDA 12, **ghi đè lên file NCCL mà torch của Colab (CUDA 13) đang dùng**. Kết quả là `import torch` hỏng (`undefined symbol: ncclCommWindowDeregister`). Mà `paddleocr` lại import torch qua `modelscope`, nên OCR cũng hỏng theo.

**Cách cell này xử lý:**
1. Kiểm tra torch còn tốt **trước khi** cài gì, và chụp danh sách thư viện `nvidia-*` của torch.
2. Cài `paddleocr==3.7.0` (ghim phiên bản, vì code viết theo API 3.x), rồi thử `paddlepaddle-gpu`.
3. **Cài lại đúng bản NCCL của torch.** Nếu torch vẫn hỏng thì cài lại toàn bộ thư viện `nvidia-*` theo danh sách đã chụp.
4. Kiểm tra ở **tiến trình riêng**: torch thấy GPU **và** Paddle chạy được trên GPU → `DEVICE = gpu`. Nếu không, gỡ Paddle GPU, khôi phục thư viện của torch, cài Paddle CPU → `DEVICE = cpu` (chậm hơn nhưng chắc chắn chạy).
5. Cuối cell **bắt buộc torch còn tốt**, nếu không thì dừng hẳn.

*Kỳ vọng: `torch OK trước khi cài`, phiên bản `paddleocr 3.7.0 | paddle …`, rồi `DEVICE = gpu` (hoặc `cpu` kèm lý do) và `✅ torch vẫn tốt`. Mất khoảng 3–6 phút.*

In [ ]:
import subprocess, json

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi chạy lại từ mục 1'
print('torch OK trước khi cài')
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()                   # thư viện CUDA của torch (chụp lại để khôi phục)

code_gpu, log_gpu = sh('pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
restore({n: v for n, v in BEFORE.items() if 'nccl' in n})     # NCCL của torch bị ghi đè -> cài lại
if not torch_ok():
    print('torch vẫn hỏng sau khi khôi phục NCCL -> khôi phục toàn bộ thư viện nvidia-* của torch')
    restore(BEFORE)

if code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    DEVICE = 'gpu'
else:
    print('⚠ Paddle GPU không chạy chung được với torch' if code_gpu == 0 else '⚠ Cài Paddle GPU lỗi:\n' + log_gpu[-400:])
    print('-> gỡ Paddle GPU, khôi phục thư viện torch, cài Paddle CPU')
    sh('pip uninstall -y -q paddlepaddle-gpu ' + ' '.join(NEW))
    restore(BEFORE)
    code_, log = sh('pip install -q paddlepaddle')
    assert code_ == 0, log
    DEVICE = 'cpu'

assert torch_ok(), '❌ torch hỏng sau khi cài Paddle -> gửi toàn bộ output cell này cho Claude'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
print('DEVICE =', DEVICE, '| ✅ torch vẫn tốt')

## 3. VietOCR + ảnh ViVQA + danh sách ảnh cần OCR

- Cài VietOCR với `--no-deps`, giống notebook 10. Gói trên PyPI khoá `pillow==10.2.0` và sẽ hạ cấp Pillow của Colab.
- Chép ảnh ViVQA từ Drive (`data/vivqa/images`) về `/content`, giống notebook 04/16. Chỉ chép ảnh còn thiếu.
- Đếm **ảnh duy nhất** của `train.json` và `test.json`, vì nhiều câu hỏi dùng chung một ảnh.
- Đọc **engine OCR của cache ViTextVQA** từ biên bản W10 (có trong git). Mục 6 sẽ bắt buộc cache ViVQA dùng **đúng** engine đó.

*Kỳ vọng: `VietOCR import OK`, bảng 2 dòng `train`/`test` (số câu, số ảnh), không có dòng `⚠ thiếu`, dòng `engine tham chiếu: PP-OCRv6_medium_det + vietocr/vgg_transformer`. Lần đầu chép ảnh mất vài phút.*

In [ ]:
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
print(sh('python -c "from vietocr.tool.predictor import Predictor; print(\'VietOCR import OK\')"')[1])

import json
from src.eval.run_baseline import prepare_images
from src.data.ocr import images_of, default_cache_path, load_ocr_cache

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vivqa'
DATA_DIR = f'{DRIVE_ROOT}/data/{DS}'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
BACKEND = 'paddledet_vietocr'                      # backend đã chọn ở W08 (luật dev300), dùng lại y hệt
C_PATH = default_cache_path(DRIVE_ROOT, DS, BACKEND)  # .../data/vivqa/ocr/paddledet_vietocr.json
OUT_DIR = f'{EXP_ROOT}/W11_ocr_cache_vivqa'
os.makedirs(OUT_DIR, exist_ok=True)
IMG = prepare_images(DS, DRIVE_ROOT)

FILES = {'train': 'train.json', 'test': 'test.json'}
IMGS, NQ = {}, {}
for k, f in FILES.items():
    IMGS[k] = images_of([f'{DATA_DIR}/{f}'], IMG)
    NQ[k] = len(json.load(open(f'{DATA_DIR}/{f}', encoding='utf-8')))
    miss = [p for p in IMGS[k].values() if not os.path.exists(p)]
    print(f'{k:6s} {NQ[k]:6d} câu | {len(IMGS[k]):6d} ảnh' + (f'  ⚠ thiếu {len(miss)} ảnh' if miss else ''))
    assert not miss, f'Thiếu ảnh {k} trên đĩa -> chạy lại cell này (ảnh sẽ được chép bù)'
assert NQ == {'train': 10799, 'test': 3001}, f'Số câu khác run C1/C3 (10799 / 3001): {NQ} -> DỪNG, báo Claude'
ALL = {**IMGS['train'], **IMGS['test']}
print(f'train ∪ test: {len(ALL)} ảnh')

REF_ENGINE = json.load(open('experiments/W10_ocr_cache_fulltrain/ocr_cache_manifest.json', encoding='utf-8'))['engine_info_w10']
print('engine tham chiếu:', REF_ENGINE['det_model'], '+', REF_ENGINE['rec_model'], '| paddleocr', REF_ENGINE['paddleocr'])
if os.path.exists(C_PATH):
    print(f'⚠ Đã có cache ViVQA ({len(load_ocr_cache(C_PATH)["images"])} ảnh) -> lần chạy trước bị ngắt; mục 5 sẽ chạy tiếp')

## 4. Chạy thử 100 ảnh test → đo tốc độ, xem tận mắt

OCR 100 ảnh đầu của `test.json`, rồi:
- tính **giây/ảnh** và **thời gian ước tính** cho phần còn lại;
- in 3 câu hỏi kèm đáp án gold và các dòng OCR đọc được, để xem tận mắt.

**Khác ViTextVQA:** ảnh ViVQA lấy từ COCO (ảnh đời thường), nhiều ảnh **không có chữ**. Vì vậy tỉ lệ ảnh có chữ thấp là **bình thường**, cell này không dừng vì lý do đó. Cell chỉ dừng nếu OCR **không chạy được ảnh nào**.

*Kỳ vọng: lần đầu tải model OCR. Sau đó `[OCR] xong`, khoảng 0,3–0,8 s/ảnh trên GPU, ước tính phần còn lại khoảng 1–2 giờ.*

In [ ]:
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files test.json --limit_images 100

c = load_ocr_cache(C_PATH)
assert c['images'] and c['timing'].get('images'), '❌ OCR chưa chạy được ảnh nào (xem lỗi phía trên) -> gửi output cell này cho Claude'
t = c['timing']; spi = t['sec'] / t['images']
with_text = sum(bool(e['lines']) for e in c['images'].values()) / len(c['images'])
print(f"\nengine: {c['engine_info']}")
print(f'{spi:.2f} s/ảnh | {100 * with_text:.0f}% ảnh có chữ (ViVQA: thấp là bình thường)')
print(f'Ước tính phần còn lại ({len(ALL) - len(c["images"])} ảnh) ≈ {spi * (len(ALL) - len(c["images"])) / 3600:.1f} giờ')

test = json.load(open(f'{DATA_DIR}/test.json', encoding='utf-8'))
shown = 0
for r in test:
    e = c['images'].get(r['image'])
    if e and e['lines'] and shown < 3:
        shown += 1
        print(f"\n— {r['question']}  | gold: {r['answers']}")
        for ln in e['lines'][:8]:
            print(f"   {ln['conf']:.2f}  {ln['text']}")

## 5. OCR toàn bộ ảnh train + test (chạy dài, tiếp tục được)

Lệnh OCR bỏ qua ảnh đã có trong cache và lưu sau mỗi 50 ảnh. Nếu Colab bị ngắt: **chạy lại mục 1 → 3**, rồi chạy lại **đúng cell này**. Không cần chạy lại mục 4.

*Kỳ vọng: `[OCR] … sẽ OCR <số ảnh còn thiếu>` rồi `[OCR] xong. Cache: …/data/vivqa/ocr/paddledet_vietocr.json`. Thời gian gần với ước tính ở mục 4.*

In [ ]:
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files train.json test.json

## 6. Kiểm tra cache + thống kê ViVQA

**Điều kiện hợp lệ** (sai một điều → cell dừng, gửi output cho Claude):
- cache phủ **đủ** ảnh train + test;
- số ảnh OCR lỗi ≤ 1%;
- engine **trùng** cache ViTextVQA ở các mục quyết định kết quả: `paddleocr`, `det_model`, `rec_model`, `vietocr_config`, `lang`. Mục `paddlepaddle` và `device` khác thì chỉ in cảnh báo và ghi vào biên bản.

**Thống kê (chỉ mô tả, không có luật):**
- `% câu có dòng OCR`: tỉ lệ câu hỏi mà prompt S2 **thực sự có thêm** dòng OCR. Phần còn lại có prompt trùng hệt C1, đây chính là phép tách hiệu ứng đã ghi trong luật W11.
- `đáp án∈OCR %` (test): đáp án gold nằm nguyên cụm trong OCR. ViVQA hỏi về màu, đồ vật, vị trí, số lượng nên con số này dự kiến **rất thấp**. Khi đó, dòng OCR gần như chỉ là **nhiễu** với ViVQA, và W12 sẽ đo xem adapter có chép nhầm chữ OCR vào đáp án không.

*Kỳ vọng: `train … ✅ | test … ✅ | lỗi 0`, `engine trùng ViTextVQA ✅`, bảng `% câu có dòng OCR` cho train/test, rồi kết quả `ocr_coverage` ghi ra `W11_ocr_cache_vivqa/ocr_coverage_test.json`.*

In [ ]:
from src.data.vivqa_dataset import load_vivqa
from src.data.ocr import attach_ocr, OCR_DEFAULTS

cache = load_ocr_cache(C_PATH)
miss = {k: sorted(set(v) - set(cache['images'])) for k, v in IMGS.items()}
n_err = sum(1 for k in ALL if cache['images'].get(k, {}).get('error'))
print(' | '.join(f"{k} {len(v) - len(miss[k])}/{len(v)} {'✅' if not miss[k] else '❌'}" for k, v in IMGS.items()) + f' | lỗi {n_err}')
assert not any(miss.values()) and n_err <= 0.01 * len(ALL), 'Thiếu ảnh hoặc lỗi > 1% -> chạy lại mục 5 / gửi output cho Claude'

eng = cache['engine_info']
MUST = ('paddleocr', 'det_model', 'rec_model', 'vietocr_config', 'lang')
bad = {k: (REF_ENGINE.get(k), eng.get(k)) for k in MUST if REF_ENGINE.get(k) != eng.get(k)}
assert not bad, f'❌ Engine OCR ViVQA khác ViTextVQA ở {bad} -> DỪNG, báo Claude'
for k in ('paddlepaddle', 'device'):
    if REF_ENGINE.get(k) != eng.get(k):
        print(f'⚠ engine_info.{k}: ViTextVQA = {REF_ENGINE.get(k)} | ViVQA = {eng.get(k)} (ghi vào biên bản)')
print('engine trùng ViTextVQA ✅')

STATS = {}
for k, f in FILES.items():
    s = load_vivqa(f'{DATA_DIR}/{f}', image_dir=IMG)
    STATS[k] = attach_ocr(s, cache, OCR_DEFAULTS['max_chars'], OCR_DEFAULTS['min_conf'])
    st = STATS[k]
    print(f"{k:6s} {st['n_samples']:6d} câu | {st['n_images']:6d} ảnh | % câu có dòng OCR {st['pct_samples_with_text']:5.1f} | "
          f"bị cắt 600 ký tự {st['pct_samples_truncated']:4.1f}% | TB {st['mean_chars']:.0f} ký tự/câu")

!python -m src.eval.ocr_coverage --data {DATA_DIR}/test.json --caches {C_PATH} --out {OUT_DIR}/ocr_coverage_test.json

## 7. Biên bản cache (để tái lập)

Ghi `experiments/W11_ocr_cache_vivqa/ocr_cache_manifest.json`: sha256 cache, engine, số ảnh, số lỗi, thống kê mục 6, thời gian OCR. File nhỏ, Cursor sẽ chép lên git. Bản thân cache nằm trên Drive. Notebook train adapter dùng chung sẽ kiểm sha256 cache trên Drive khớp biên bản này trước khi train.

*Kỳ vọng: 1 dòng tóm tắt (số ảnh, s/ảnh, sha256 12 ký tự đầu) + đường dẫn biên bản. Sau đó **Runtime → Disconnect and delete runtime** để không tốn thêm compute.*

In [ ]:
import hashlib, datetime
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

man = {'what': 'OCR cache VietOCR cho ViVQA train + test (W11, adapter dùng chung S2) — file cache nằm trên Drive',
       'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
       'cache_file': C_PATH.replace(DRIVE_ROOT + '/', ''), 'sha256': sha256_file(C_PATH),
       'backend': cache['backend'], 'engine_info': eng, 'reference_engine_vitextvqa_w10': REF_ENGINE,
       'engine_must_match': list(MUST), 'device': DEVICE,
       'n_images': len(cache['images']), 'n_errors': n_err,
       'pct_images_with_text': round(100 * sum(bool(e.get('lines')) for e in cache['images'].values()) / len(cache['images']), 2),
       'images_needed': {k: len(v) for k, v in IMGS.items()}, 'questions': NQ,
       'prompt_stats_v1_600_0': STATS, 'timing': cache['timing']}
json.dump(man, open(f'{OUT_DIR}/ocr_cache_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(f"{man['n_images']} ảnh | lỗi {n_err} | {man['timing']['sec'] / man['timing']['images']:.2f} s/ảnh | "
      f"{man['pct_images_with_text']}% ảnh có chữ | sha256 {man['sha256'][:12]} | device {DEVICE}")
print('->', f'{OUT_DIR}/ocr_cache_manifest.json')